In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# ---------------- Feature Importance ----------------
def featureImportance(indep_X, dep_Y, n):  # select top n features using feature importance

    #Random Forest is used because it can automatically measure feature importance based on how much each feature helps improve prediction accuracy.
    model = RandomForestClassifier(n_estimators=100, random_state=0)  # create RF model
    model.fit(indep_X, dep_Y)  # train model
    
    importances = model.feature_importances_  # get importance scores
    
    feature_names = indep_X.columns  # get column names
    
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})  # create table
    
    importance_df = importance_df.sort_values(by='Importance', ascending=False)  # sort highest first
    
    selected_features = importance_df['Feature'].head(n)  # select top n features
    
    return indep_X[selected_features]  # return dataset with selected features


# ---------------- Split + Scale ----------------
def split_scalar(indep_X, dep_Y):  # split and scale data
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
    
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    
    return X_train, X_test, y_train, y_test


# ---------------- Evaluation ----------------
def cm_prediction(classifier, X_test, y_test):  # evaluate model
    y_pred = classifier.predict(X_test)
    
    from sklearn.metrics import accuracy_score
    
    Accuracy = accuracy_score(y_test, y_pred)
    
    return Accuracy


# ---------------- Models ----------------
def logistic(X_train, y_train, X_test, y_test):  # logistic model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def svm_linear(X_train, y_train, X_test, y_test):  # linear svm
    model = SVC(kernel='linear')
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def svm_NL(X_train, y_train, X_test, y_test):  # nonlinear svm
    model = SVC(kernel='rbf')
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def knn(X_train, y_train, X_test, y_test):  # knn
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def Navie(X_train, y_train, X_test, y_test):  # naive bayes
    model = GaussianNB()
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def Decision(X_train, y_train, X_test, y_test):  # decision tree
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)

def random(X_train, y_train, X_test, y_test):  # random forest
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X_train, y_train)
    return cm_prediction(model, X_test, y_test)


# ---------------- Result Table ----------------
def fi_classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf):  # create result table
    
    df = pd.DataFrame(index=['FeatureImportance'], columns=['Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random'])
    
    df.loc['FeatureImportance','Logistic'] = acclog[0]
    df.loc['FeatureImportance','SVMl'] = accsvml[0]
    df.loc['FeatureImportance','SVMnl'] = accsvmnl[0]
    df.loc['FeatureImportance','KNN'] = accknn[0]
    df.loc['FeatureImportance','Navie'] = accnav[0]
    df.loc['FeatureImportance','Decision'] = accdes[0]
    df.loc['FeatureImportance','Random'] = accrf[0]
    
    return df

In [2]:
dataset1 = pd.read_csv("prep.csv")  # load dataset
df2 = pd.get_dummies(dataset1, drop_first=True)  # convert categorical

indep_X = df2.drop(columns=['classification_yes'])  # features
dep_Y = df2['classification_yes']  # target

In [3]:
selected_X = featureImportance(indep_X, dep_Y, 5)  # select top 5 features

X_train, X_test, y_train, y_test = split_scalar(selected_X, dep_Y)

# store accuracy
acclog=[]; accsvml=[]; accsvmnl=[]; accknn=[]; accnav=[]; accdes=[]; accrf=[]

acclog.append(logistic(X_train,y_train,X_test,y_test))
accsvml.append(svm_linear(X_train,y_train,X_test,y_test))
accsvmnl.append(svm_NL(X_train,y_train,X_test,y_test))
accknn.append(knn(X_train,y_train,X_test,y_test))
accnav.append(Navie(X_train,y_train,X_test,y_test))
accdes.append(Decision(X_train,y_train,X_test,y_test))
accrf.append(random(X_train,y_train,X_test,y_test))

In [4]:
result = fi_classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf)
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
FeatureImportance,0.94,0.97,0.96,0.94,0.88,0.96,0.97
